<a href="https://colab.research.google.com/github/Jiaze-G/resume-hiring-prediction-ml/blob/main/Modeling_and_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modeling and Evaluation
## Overview:
This file aims to build different models and evaluate them.
## Purpose:

## Setup

In [9]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [2]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

DATA_DIR = "/content/drive/MyDrive/EE344/final project/processed"

# Load the data
X_train = pd.read_parquet(f"{DATA_DIR}/X_train.parquet")
X_test  = pd.read_parquet(f"{DATA_DIR}/X_test.parquet")

y_train = pd.read_parquet(f"{DATA_DIR}/y_train.parquet")["target"]
y_test  = pd.read_parquet(f"{DATA_DIR}/y_test.parquet")["target"]

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("\nTraining label distribution:")
print(y_train.value_counts())

Training feature shape: (800, 115)
Testing feature shape: (200, 115)

Training label distribution:
target
Hire      650
Reject    150
Name: count, dtype: int64


**Preprocessing and Scaling**

In [16]:
#Encode target variable(Hire=1, Reject=0)
y_train_num = y_train.map({'Hire': 1, 'Reject': 0})
y_test_num = y_test.map({'Hire': 1, 'Reject': 0})

#Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Applying Models**

In [12]:
RANDOM_STATE = 42
lr_pipe = Pipeline(steps=[
    ("scaler", scaler),
    ("model", LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
])

rf_pipe = Pipeline(steps=[
    ("scaler", scaler), # RF doesn't strictly need scaling, but it keeps pipelines consistent
    ("model", RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
])

svm_pipe = Pipeline(steps=[
    ("scaler", scaler),
    ("model", SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE))
])

dt_pipe = Pipeline(steps=[
    ("scaler", scaler),
    ("model", DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE))
])

MODELS = {
    "LogisticRegression": lr_pipe,
    "RandomForest": rf_pipe,
    "SVM": svm_pipe,
    "DecisionTree": dt_pipe
}

MODELS

{'LogisticRegression': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  LogisticRegression(class_weight='balanced', max_iter=1000,
                                     random_state=42))]),
 'RandomForest': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  RandomForestClassifier(class_weight='balanced',
                                         n_estimators=200, n_jobs=-1,
                                         random_state=42))]),
 'SVM': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  SVC(class_weight='balanced', probability=True,
                      random_state=42))]),
 'DecisionTree': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  DecisionTreeClassifier(class_weight='balanced',
                                         random_state=42))])}

**Evaluation**

In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
def evaluate_classification(model, X_train, X_test, y_train, y_test):
    """Fit on train, evaluate on train and test using classification metrics."""
    pred_train = model.predict(X_train)
    pred_test  = model.predict(X_test)

    results = {
        "F1_train": f1_score(y_train, pred_train),
        "F1_test":  f1_score(y_test, pred_test),
        "Precision_test": precision_score(y_test, pred_test),
        "Recall_test": recall_score(y_test, pred_test),
        "Accuracy_test": accuracy_score(y_test, pred_test)
    }

    return results

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [24]:
# === Quick Fix: Ensure labels are numeric (Hire = 1, Reject = 0) ===
y_train = y_train.replace({'Hire': 1, 'Reject': 0})
y_test = y_test.replace({'Hire': 1, 'Reject': 0})

rows = []

for name, pipe in MODELS.items():
    # --- CV F1-Score ---
    # Passing cv=5 automatically uses StratifiedKFold for classification tasks!
    cv_scores = cross_val_score(
        pipe,
        X_train, y_train,
        scoring="f1",
        cv=5,
        n_jobs=-1
    )
    cv_f1 = cv_scores.mean()

    # --- Test metrics ---
    pipe.fit(X_train, y_train)
    metrics = evaluate_classification(pipe, X_train, X_test, y_train, y_test)

    rows.append({
        "Model": name,
        "CV F1-Score": cv_f1,
        "Test F1": metrics["F1_test"],
        "Test Precision": metrics["Precision_test"],
        "Test Recall": metrics["Recall_test"],
        "Test Accuracy": metrics["Accuracy_test"]
    })

# Collect results into a clean dataframe
results_df = pd.DataFrame(rows).sort_values("Test F1", ascending=False).reset_index(drop=True)
display(results_df.round(4))

,Model,CV F1-Score,Test F1,Test Precision,Test Recall,Test Accuracy
0,RandomForest,0.9992,1.0000,1.0000,1.0000,1.000
1,DecisionTree,1.0000,1.0000,1.0000,1.0000,1.000
2,LogisticRegression,0.9689,0.9812,0.9937,0.9691,0.970
3,SVM,0.9396,0.9590,0.9806,0.9383,0.935


In [ ]:
f1_scores = {}
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(16, 6))